# **Chapter 29: Magpie**

## **Section 1: 事前学習モデルの読み込み**

In [ ]:
# 学習ハイパーパラメータ
INPUT_SEQUENCE_LENGTH = 1024
BATCH_SIZE = 1
GLOBAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = GLOBAL_BATCH_SIZE // BATCH_SIZE
MAX_LEARNING_RATE = 1e-4
MIN_LEARNING_RATE = 1e-5
WARMUP_STEPS = 1_000
MAX_GRAD_NORM = 1.0
ADAM_BETAS = (0.9, 0.95)
WEIGHT_DECAY = 0.0
NUM_EPOCHS = 1
DEVICE = "cuda"
#UPLOAD_TO_HUB = False
#HF_REPO_ID = "your-account/qwen3-instruction-tuned"


In [ ]:
import torch
import random
RANDOM_SEED = 1337
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    torch_dtype=torch.bfloat16,
).to(DEVICE)
model.config.use_cache = False

print("Qwen3-0.6B loaded")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen3-0.6B loaded


In [ ]:
# ⚠️ Don't run this cell twice!
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
compiled_model = torch.compile(model)

In [ ]:
compiled_model = compiled_model.to(DEVICE)
compiled_model.eval()

OptimizedModule(
  (_orig_mod): Qwen3ForCausalLM(
    (model): Qwen3Model(
      (embed_tokens): Embedding(151936, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
            (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
            (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          )
          (mlp): Qwen3MLP(
            (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
            (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
            (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
            (act_fn): SiLUActivation()
          )
          (input_laye

期待する形式

以下のデータセットと同じ形式

```plain
{"prompt": "What is the capital of Japan?", "response": "The capital of Japan is Tokyo."}
{"prompt": "Hello there!", "response": "Hello! How can I assist you today?"}
{"prompt": "What is the result of 2^3?", "response": "The result of 2^3 is 8."}
```

# 入れ替えポイント2
# 他のデータが良ければ、ここを入れ替えてね

In [20]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="HayatoHongo/Magpie-Phi3-Pro-1M-v0.1",
    repo_type="dataset",
    filename="sft_prompt_response_phi3.jsonl",
    local_dir=".",
)

'/content/sft_prompt_response_phi3.jsonl'

80万サンプル規模のデータセットでL4GPU付属CPUインスタンスなら20分かかる

In [21]:
import json

jsonl_file = open("/content/sft_prompt_response_phi3.jsonl", "r", encoding="utf-8")
output_jsonl_file = open("/content/sft_token_pairs.jsonl", "w", encoding="utf-8")

for json_line in jsonl_file:
    sample = json.loads(json_line)

    prompt_text = "<USER>" + sample["prompt"] + "<ASSISTANT>"
    response_text = sample["response"] + tokenizer.eos_token

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(response_text, add_special_tokens=False)["input_ids"]

    input_ids = (prompt_ids + response_ids)[:INPUT_SEQUENCE_LENGTH]
    target_ids = ([-100] * len(prompt_ids) + response_ids)[:INPUT_SEQUENCE_LENGTH]

    padding = INPUT_SEQUENCE_LENGTH - len(input_ids)
    input_ids += [tokenizer.pad_token_id] * padding
    target_ids += [-100] * padding

    output_jsonl_file.write(json.dumps({
        "padded_input_ids": input_ids,
        "padded_target_ids": target_ids,
    }) + "\n")

jsonl_file.close()
output_jsonl_file.close()


In [22]:
padded_input_ids_list = []
jsonl_file = open("/content/sft_token_pairs.jsonl", "r", encoding="utf-8")

for json_line in jsonl_file:
    sample = json.loads(json_line)
    padded_input_ids_list.append(sample["padded_input_ids"])

input_ids_tensor = torch.tensor(padded_input_ids_list, dtype=torch.long)
jsonl_file.close()

torch.save(input_ids_tensor, "/content/input_ids.pt")
print("input_ids:", input_ids_tensor.shape)

input_ids: torch.Size([803975, 1024])


In [23]:
# CPUメモリを解放するために、不要な変数を削除してガベージコレクションを実行します。
import gc
del padded_input_ids_list, input_ids_tensor
gc.collect()

32

In [24]:
padded_target_ids_list = []
jsonl_file = open("/content/sft_token_pairs.jsonl", "r", encoding="utf-8")

for json_line in jsonl_file:
    sample = json.loads(json_line)
    padded_target_ids_list.append(sample["padded_target_ids"])

target_ids_tensor = torch.tensor(padded_target_ids_list, dtype=torch.long)
jsonl_file.close()

torch.save(target_ids_tensor, "/content/target_ids.pt")
print("target_ids:", target_ids_tensor.shape)

target_ids: torch.Size([803975, 1024])


In [25]:
# CPUメモリを解放するために、不要な変数を削除してガベージコレクションを実行します。
import gc
del padded_target_ids_list, target_ids_tensor
gc.collect()

9

In [ ]:
# Google Driveに保存. 次回から再利用できる
from google.colab import drive
drive.mount('/content/drive')
!cp /content/input_ids.pt /content/drive/MyDrive/instructiontuningqwen3_0.6b/input_ids.pt
!cp /content/target_ids.pt /content/drive/MyDrive/instructiontuningqwen3_0.6b/target_ids.pt
print('input_ids.pt と target_ids.pt を MyDrive に保存しました')


In [26]:
class DataLoader:
    def __init__(self, input_ids_tensor_path, target_ids_tensor_path):
        self.input_ids = torch.load(input_ids_tensor_path, mmap=True)
        self.target_ids = torch.load(target_ids_tensor_path, mmap=True)
        self.data_size = len(self.input_ids)
        self.current_index = 0

    def get_batch(self):
        start_index = self.current_index
        end_index = start_index + BATCH_SIZE

        if end_index > self.data_size:
            start_index = 0
            end_index = BATCH_SIZE

        input_batch = self.input_ids[start_index:end_index]
        target_batch = self.target_ids[start_index:end_index]
        self.current_index = end_index

        input_batch = input_batch.to(DEVICE)
        target_batch = target_batch.to(DEVICE)

        return input_batch, target_batch

In [27]:
data_loader = DataLoader(
    input_ids_tensor_path="/content/input_ids.pt",
    target_ids_tensor_path="/content/target_ids.pt",
)

In [28]:
import time


class Trainer:
    def __init__(self, model, optimizer, data_loader):
        self.model = model
        self.optimizer = optimizer
        self.data_loader = data_loader
        self.steps = []
        self.learning_rates = []
        self.train_losses = []
        self.tokens_per_second_list = []

    def train_step(self):
        self.optimizer.zero_grad(set_to_none=True)
        loss_sum = 0.0

        for _ in range(GRADIENT_ACCUMULATION_STEPS):
            input_batch, target_batch = self.data_loader.get_batch()
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                loss = self.model(input_ids=input_batch, labels=target_batch).loss
            (loss / GRADIENT_ACCUMULATION_STEPS).backward()
            loss_sum += loss.item()

        torch.nn.utils.clip_grad_norm_(
            self.model.parameters(),
            MAX_GRAD_NORM,
        )
        self.optimizer.step()
        return loss_sum / GRADIENT_ACCUMULATION_STEPS

    def train(self):
        self.model.train()
        last_log_time = time.time()

        for step in range(1, TOTAL_STEPS + 1):
            learning_rate = get_learning_rate(step)
            self.optimizer.param_groups[0]["lr"] = learning_rate
            train_loss = self.train_step()

            now = time.time()
            interval = now - last_log_time
            tokens = GLOBAL_BATCH_SIZE * INPUT_SEQUENCE_LENGTH
            tokens_per_second = tokens / interval if interval > 0 else None

            print(
                f"step {step:05d} | "
                f"lr {learning_rate:.6e} | "
                f"train loss {train_loss:.4f} | "
                f"tok/s {tokens_per_second}"
            )
            self.steps.append(step)
            self.learning_rates.append(learning_rate)
            self.train_losses.append(train_loss)
            self.tokens_per_second_list.append(tokens_per_second)
            last_log_time = now


In [29]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=MAX_LEARNING_RATE,
    betas=ADAM_BETAS,
    weight_decay=WEIGHT_DECAY,
    fused=True,
)


In [30]:
compiled_model.train()

OptimizedModule(
  (_orig_mod): Qwen3ForCausalLM(
    (model): Qwen3Model(
      (embed_tokens): Embedding(151936, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
            (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
            (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          )
          (mlp): Qwen3MLP(
            (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
            (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
            (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
            (act_fn): SiLUActivation()
          )
          (input_laye

In [31]:
# GLOBAL_BATCH_SIZE と BATCH_SIZE から自動計算されます。

In [32]:
def get_learning_rate(step):
    if step <= WARMUP_STEPS:
        return MAX_LEARNING_RATE * step / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return MIN_LEARNING_RATE + (MAX_LEARNING_RATE - MIN_LEARNING_RATE) * (1 - progress)


In [33]:
data_size = data_loader.data_size
TOTAL_STEPS = data_size * NUM_EPOCHS // GLOBAL_BATCH_SIZE
print("total steps:", TOTAL_STEPS)

total steps: 25124


In [34]:
trainer = Trainer(
    model=compiled_model,
    optimizer=optimizer,
    data_loader=data_loader,
)


In [ ]:
trainer.train()

W0916 19:55:33.153000 5654 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


step 00001 | lr 1.000000e-07 | train loss 1.7827 | tok/s 361.2308049158052
step 00002 | lr 2.000000e-07 | train loss 1.4834 | tok/s 6163.16297654339
step 00003 | lr 3.000000e-07 | train loss 1.2290 | tok/s 6064.098479873715
step 00004 | lr 4.000000e-07 | train loss 1.4141 | tok/s 6054.682937840524
step 00005 | lr 5.000000e-07 | train loss 1.4655 | tok/s 6051.074090207655
step 00006 | lr 6.000000e-07 | train loss 1.2016 | tok/s 6037.10773916632
step 00007 | lr 7.000000e-07 | train loss 2.1779 | tok/s 6078.124280809888
step 00008 | lr 8.000000e-07 | train loss 1.2710 | tok/s 6075.253509033737
step 00009 | lr 9.000000e-07 | train loss 1.4640 | tok/s 6089.626455552101
step 00010 | lr 1.000000e-06 | train loss 1.3886 | tok/s 6134.1479948560955
step 00011 | lr 1.100000e-06 | train loss 1.3147 | tok/s 6151.298179154433
step 00012 | lr 1.200000e-06 | train loss 1.5844 | tok/s 6152.749961824815
step 00013 | lr 1.300000e-06 | train loss 1.3646 | tok/s 6142.038834212131
step 00014 | lr 1.400000e-

In [2]:
compiled_model.eval()
prompt = "<USER>Explain the concept of gravity.<ASSISTANT>"
encoded = tokenizer(prompt, add_special_tokens=False)["input_ids"]
input_tensor = torch.tensor([encoded], dtype=torch.long, device=DEVICE)
generated_tensor = compiled_model.generate(
    input_tensor,
    max_new_tokens=64,
    temperature=0.5,
)
print(tokenizer.decode(generated_tensor[0].tolist()))


NameError: name 'compiled_model' is not defined

In [ ]:
# グラフ描画。
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.plot(trainer.steps, trainer.train_losses, label='Train Loss')

plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training and Validation Loss over Steps')
plt.legend()
plt.grid(True)
plt.show()